In [ ]:
def save_dataframe_to_postgresql(
    engine, data, load_behavior: str, default_table: str, history_table: str = None
):
    """
    Save pd.DataFrame to psql.
    A high-level API for pd.DataFrame.to_psql with `index=False` and `schema='public'`.
    If the data is gpd.GeoDataFrame, use function `save_geodataframe_to_postgresql` instead.

    Args:
    engine : sqlalchemy.engine.base.Engine.
    data : pd.DataFrame. Data to be saved.
    load_behavior : str. Save mode, should be one of `append`, `replace`, `current+history`.
        `append`: Just append new data to the `default_table`.
        `replace`: Truncate the `default_table` and append new data.
        `current+history`: The current+history design is intended to preserve historical records
            while simultaneously maintaining the most recent data. This proces will truncate
            the `default_table` and append the new data into it, then append the new data
            to `history_table`.
    default_table : str. Default table name.
    history_table : str. History table name, only used when load_behavior is `current+history`.

    Use case:
    # original
    table_name = 'test_123'
    with engine.connect() as conn:
        sql = f'TRUNCATE TABLE {table_name}'
        conn.execute(sa_text(sql).execution_options(autocommit=True))
        mapping_table.to_sql(table_name, conn, if_exists='append', index=False, schema='public')
        conn.commit()
    # function
    save_dataframe_to_postgresql(
        engine, data=mapping_table, load_behavior='replace', current_table='test_123',
    )
    """
    # check data type
    if isinstance(data, gpd.GeoDataFrame):
        raise ValueError(
            "Data type is gpd.GeoDataFrame, use function `save_geodataframe_to_postgresql` instead."
        )

    is_column_include_geometry = data.columns.isin(["wkb_geometry", "geometry"])
    if any(is_column_include_geometry):
        raise ValueError(
            """
            Column name contains `wkb_geometry` or `geometry`, it should be a GeoDataFrame.
            Please use function `save_geodataframe_to_postgresql` instead.
        """
        )

    start_time = time.time()

    # main
    conn = engine.connect()
    if load_behavior == "append":
        data.to_sql(
            default_table, conn, if_exists="append", index=False, schema="public"
        )
    elif load_behavior == "replace":
        conn.execute(
            sa_text(f"TRUNCATE TABLE {default_table}").execution_options(
                autocommit=True
            )
        )
        data.to_sql(
            default_table, conn, if_exists="append", index=False, schema="public"
        )
    elif load_behavior == "current+history":
        if history_table is None:
            raise ValueError(
                "history_table should be provided when load_behavior is `current+history`."
            )
        conn.execute(
            sa_text(f"TRUNCATE TABLE {default_table}").execution_options(
                autocommit=True
            )
        )
        data.to_sql(
            default_table, conn, if_exists="append", index=False, schema="public"
        )
        data.to_sql(
            history_table, conn, if_exists="append", index=False, schema="public"
        )
    else:
        raise ValueError(
            "load_behavior should be one of `append`, `replace`, `current+history`."
        )
    conn.close()

    # print
    cost_time = time.time() - start_time
    print(f"Data been saved, cost time: {cost_time:.2f}s.")


In [14]:
from datetime import datetime, timedelta

import pytz

TAIPEI_TZ = pytz.timezone("Asia/Taipei")

def get_tpe_now_time_str(is_with_tz=False):
    """
    This function retrieves the current date and time in Taipei, Taiwan. It offers options to
    include or exclude the timezone information. The output is a string.

    Args:
        is_with_tz (bool, optional): _description_. Defaults to False.

    Example
    ----------
    tpe_now_utc = get_tpe_now_time_str()
    print(tpe_now_utc)
    print(type(tpe_now_utc))
    # Output: 2024-04-26 17:55:00 (without timezone)
    # Output: <class 'str'>
    """
    if is_with_tz:
        return str(datetime.now(tz=TAIPEI_TZ).replace(microsecond=0))
    else:
        return str(datetime.now().replace(microsecond=0))


In [17]:
import geopandas as gpd
import pandas as pd
import pyproj
from geoalchemy2 import WKTElement
from numpy import nan
from shapely.geometry import MultiLineString
from shapely.geometry.multipolygon import MultiPolygon
from shapely.geometry.polygon import Polygon


def add_point_wkbgeometry_column_to_df(
    data: pd.DataFrame,
    x: pd.Series,
    y: pd.Series,
    from_crs: int,
    to_crs=4326,
    is_add_xy_columns=True,
) -> gpd.GeoDataFrame:
    """
    Convert original DataFrame with x and y to GeoDataFrame with wkbgeometry.
    Input should be a pandas.DataFrame.
    Output will be a geopandas.GeoDataFrame and add 3 columns - wkb_geometry, lng, lat.

    Parameters
    ----------
    df: input DataFrame.
    x: x or lng.
    y: y or lat.
    from_crs: crs alias name as EPSG, commonly use 4326=WGS84 or 3826=TWD97
    to_crs: crs alias name as EPSG, default 4326.
    is_add_xy_columns: Add lng(x), lat(y) to output, defalut True.
        Only point type geometry will add column.
        Add these two column can benifit powerBI user.

    Example
    ----------
    x = pd.Series([262403.2367, 481753.6091, '', None])
    y = pd.Series([2779407.0527, 2914189.1837, None, ''])
    xx = pd.to_numeric(x, errors='coerce')
    gdf = add_point_wkbgeometry_column_to_df(data, x, y, from_crs=3826)
    gdf = add_point_wkbgeometry_column_to_df(data, x, y, from_crs=3826, is_add_xy_columns=False)
    """
    # covert column type
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    geometry = gpd.points_from_xy(x, y)

    # make df to gdf
    df = data.copy()
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=f"EPSG:{from_crs}")
    if from_crs == to_crs:
        gdf = gdf.to_crs(epsg=from_crs)
        gdf = gdf.to_crs(epsg=to_crs)
    else:
        gdf = gdf.to_crs(epsg=to_crs)

    # add column
    if is_add_xy_columns:
        geo_type = gdf["geometry"].type.iloc[0]
        if geo_type == "Point":
            gdf["lng"] = gdf["geometry"].map(
                lambda ele: ele.x if not ele.is_empty else nan
            )
            gdf["lat"] = gdf["geometry"].map(
                lambda ele: ele.y if not ele.is_empty else nan
            )
    gdf["wkb_geometry"] = gdf["geometry"].apply(
        lambda x: WKTElement(x.wkt, srid=to_crs) if x is not None else None
    )

    return gdf

In [21]:
import requests
import pandas as pd
from io import StringIO
from sqlalchemy import create_engine


csv_path = "/Users/linwebber/Desktop/cooling.csv"

# 你的 cooling.csv 看起來不是 UTF-8；pandas 讀 CSV 應該用 encoding 參數，而不是對 DataFrame 做 decode()
try:
    raw_data = pd.read_csv(csv_path, encoding="big5", encoding_errors="strict")
except UnicodeDecodeError:
    # Windows 常見的繁中 CSV 編碼
    raw_data = pd.read_csv(csv_path, encoding="cp950", encoding_errors="replace")

# Transform
data = raw_data.copy()

data


,編號,設施地點\n（戶外或室內）,名稱,行政區,地址,經度,緯度,市話,分機,手機,其他聯絡方式,開放時間,電風扇,冷氣,廁所,座位,飲水設施（例如：飲水機；直飲台；奉茶點等）,無障礙座位,其他特色及亮點,備註
0,1,戶外,線形公園7號(士林),士林區,基隆河北-捷運劍潭站南段(界線:基隆河-通河街),121.5234196,25.07922114,(02)28812912,NaN,NaN,NaN,24小時,N,N,N,Y,N,N,NaN,高架橋下
1,2,戶外,線形公園8號(士林),士林區,基隆河北-捷運劍潭站南段(界線:通河街-基河路),121.5243412,25.08190178,(02)28812912,NaN,NaN,NaN,24小時,N,N,N,Y,N,N,NaN,高架橋下
2,3,戶外,線形公園9號(士林),士林區,捷運劍潭站北-士林站南段\n(界線:文林、基河路口-中山北路5段461巷),121.5271819,25.0884807,(02)28812912,NaN,NaN,NaN,24小時,N,N,N,Y,N,N,NaN,高架橋下
3,4,戶外,線形公園10號(士林),士林區,捷運士林站北-雙溪南段(界線:中正路-前街),121.5252553,25.09620154,(02)28812912,NaN,NaN,NaN,24小時,N,N,N,Y,N,N,NaN,高架橋下
4,5,戶外,線形公園11號(士林),士林區,捷運士林站北-雙溪南段(界線:前街-後街),121.5246077,25.09779632,(02)28812912,NaN,NaN,NaN,24小時,N,N,N,Y,N,N,NaN,高架橋下，設有兒童遊戲場
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
489,490,戶外,莒光社會住宅公共空間,萬華區,萬大路22號,121.5,25.03195,(02)25925116,NaN,NaN,NaN,24小時,N,N,N,N,N,N,NaN,NaN
490,491,戶外,福星社會住宅公共空間,萬華區,開封街二段39號,121.5065,25.04701,(02)25925116,NaN,NaN,NaN,24小時,N,N,N,N,N,N,NaN,NaN
491,492,室內,萬華運動中心,萬華區,西寧南路6-1號,121.507142,25.047498,(02)23759900,NaN,NaN,NaN,6:00-22:00,Y,Y,Y,Y,Y,N,NaN,NaN
492,493,室內,臺北市青年公園運動休閒園區,萬華區,水源路199號,121.506975,25.022082,(02)23053800,NaN,NaN,NaN,6:00-22:00,Y,N,Y,Y,N,N,NaN,NaN


In [24]:

# Transform: rename fields to standardized columns
data = data.rename(
	columns={
		"編號": "id",
		"設施地點\n（戶外或室內）": "location_type",
		"名稱": "name",
		"行政區": "area",
		"地址": "address",
		"經度": "longitude",
		"緯度": "latitude",
		"市話": "localcall",
		"分機": "ext",
		"手機": "mobile",
		"其他聯絡方式": "contact_other",
		"開放時間": "open_time",
		"電風扇": "fan",
		"冷氣": "aircon",
		"廁所": "toilet",
		"座位": "seat",
		"飲水設施（例如：飲水機；直飲台；奉茶點等）": "water_facility",
		"無障礙座位": "accessible_seat",
		"其他特色及亮點": "features",
		"備註": "note",
	}
 )

# 有些資料會把座標塞在同一欄，用逗號分隔；先把全形逗號統一成半形逗號
data["longitude"] = data["longitude"].astype(str).str.replace("，", ",").str.strip()
mask = data["longitude"].str.contains(",", na=False)
if mask.any():
	split_coords = data.loc[mask, "longitude"].str.split(",", expand=True)
	# 若出現逗號，取最後一段（通常是經度），並去除空白
	data.loc[mask, "longitude"] = split_coords.iloc[:, -1].astype(str).str.strip()

data["longitude"] = pd.to_numeric(data["longitude"], errors="coerce")
data["latitude"] = pd.to_numeric(data["latitude"], errors="coerce")
data["data_time"] = get_tpe_now_time_str(is_with_tz=True)
FROM_CRS = 4326
# standardize geometry
gdata = add_point_wkbgeometry_column_to_df(
	data, x=data["longitude"], y=data["latitude"], from_crs=FROM_CRS
)

# gdata
# select column
ready_data = gdata[["data_time", "id", "location_type", "name", "area", "address", "longitude", "latitude", "localcall", "ext", "mobile", "contact_other", "open_time", "fan", "aircon", "toilet", "seat", "water_facility", "accessible_seat", "features", "note", "wkb_geometry"]]

In [25]:
ready_data

,data_time,id,location_type,name,area,address,longitude,latitude,localcall,ext,...,open_time,fan,aircon,toilet,seat,water_facility,accessible_seat,features,note,wkb_geometry
0,2025-12-22 11:08:13+08:00,1,戶外,線形公園7號(士林),士林區,基隆河北-捷運劍潭站南段(界線:基隆河-通河街),121.523420,25.079221,(02)28812912,NaN,...,24小時,N,N,N,Y,N,N,NaN,高架橋下,POINT (121.5234196 25.07922114)
1,2025-12-22 11:08:13+08:00,2,戶外,線形公園8號(士林),士林區,基隆河北-捷運劍潭站南段(界線:通河街-基河路),121.524341,25.081902,(02)28812912,NaN,...,24小時,N,N,N,Y,N,N,NaN,高架橋下,POINT (121.5243412 25.08190178)
2,2025-12-22 11:08:13+08:00,3,戶外,線形公園9號(士林),士林區,捷運劍潭站北-士林站南段\n(界線:文林、基河路口-中山北路5段461巷),121.527182,25.088481,(02)28812912,NaN,...,24小時,N,N,N,Y,N,N,NaN,高架橋下,POINT (121.5271819 25.0884807)
3,2025-12-22 11:08:13+08:00,4,戶外,線形公園10號(士林),士林區,捷運士林站北-雙溪南段(界線:中正路-前街),121.525255,25.096202,(02)28812912,NaN,...,24小時,N,N,N,Y,N,N,NaN,高架橋下,POINT (121.5252553 25.09620154)
4,2025-12-22 11:08:13+08:00,5,戶外,線形公園11號(士林),士林區,捷運士林站北-雙溪南段(界線:前街-後街),121.524608,25.097796,(02)28812912,NaN,...,24小時,N,N,N,Y,N,N,NaN,高架橋下，設有兒童遊戲場,POINT (121.5246077 25.09779632)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
489,2025-12-22 11:08:13+08:00,490,戶外,莒光社會住宅公共空間,萬華區,萬大路22號,121.500000,25.031950,(02)25925116,NaN,...,24小時,N,N,N,N,N,N,NaN,NaN,POINT (121.5 25.03195)
490,2025-12-22 11:08:13+08:00,491,戶外,福星社會住宅公共空間,萬華區,開封街二段39號,121.506500,25.047010,(02)25925116,NaN,...,24小時,N,N,N,N,N,N,NaN,NaN,POINT (121.5065 25.04701)
491,2025-12-22 11:08:13+08:00,492,室內,萬華運動中心,萬華區,西寧南路6-1號,121.507142,25.047498,(02)23759900,NaN,...,6:00-22:00,Y,Y,Y,Y,Y,N,NaN,NaN,POINT (121.507142 25.047498)
492,2025-12-22 11:08:13+08:00,493,室內,臺北市青年公園運動休閒園區,萬華區,水源路199號,121.506975,25.022082,(02)23053800,NaN,...,6:00-22:00,Y,N,Y,Y,N,N,NaN,NaN,POINT (121.506975 25.022082)


In [26]:
ready_data.to_csv('/Users/linwebber/Desktop/cooling_cleaned.csv', index=False)